In [1]:

from transformers import pipeline
from dotenv import load_dotenv
import os

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")

classifier = pipeline(
    task="zero-shot-classification",
    model="facebook/bart-large-mnli",
    token=HF_TOKEN
)



def predict_sentiment(text):
    result = sentiment_pipeline(text)[0]

    return {
        "sentiment": result["label"],
        "confidence": round(result["score"], 4)
    }

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [2]:
result = classifier(
    "The fruit and vegetables keep going bad before the expiry date.",
    candidate_labels=[
        "positive",
        "negative",
        "neutral"
    ]
)

for label,score in zip(result["labels"],result["scores"]):
    print(f"{label:20} {score:.2%}") 
#print(result)

negative             96.38%
neutral              2.50%
positive             1.12%


In [3]:
import sentiment

print(sentiment.__file__)

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

/workspaces/codespaces-jupyter/codebase/sentiment.py


In [6]:
from sentiment import predict_sentiment

result = predict_sentiment(
    "The fruit and vegetables keep going bad before the expiry date."
)

print(result)

{'sentiment': 'negative', 'confidence': 0.9638}


In [5]:
import pandas as pd
from sentiment import predict_sentiment

df = pd.read_csv(
    "product_feedback.csv"
)

results = df["Comments"].apply(
    predict_sentiment
)

df["Sentiment"] = results.apply(
    lambda x: x["sentiment"]
)

df["Confidence"] = results.apply(
    lambda x: x["confidence"]
)

print(df.head())

                                        Product Name  Product ID  \
0      ASDA Simple to Cook Butter Chicken Curry 350g   100532576   
1                       Asda Scotch Eggs Choriz 226G   100551875   
2   ASDA Free From Free From by 2 Cod Fishcakes 180g   100165747   
3  Exceptional by ASDA Exceptional by Chicken Gra...   100119358   
4  JUST ESSENTIALS by ASDA Cooked & Peeled Prawns...   100169178   

                      Category How likely are you to recommend this product?  \
0                READY TO COOK                                           NaN   
1                  SCOTCH EGGS                                           NaN   
2                    FREE FROM                                           NaN   
3  GRAVIES, STOCKS & STUFFINGS                                             7   
4                       I-FISH                                             7   

  Complete - Yes/No. If No, please provide rationale  \
0                    No - Store/Online doesn't stock  

In [7]:
df["Comments"] = (
    df["Comments"]
    .fillna("")
    .astype(str)
    .str.strip()
)

df = df[
    df["Comments"] != ""
]

In [8]:
import gradio as gr
from sentiment import predict_sentiment

def analyse(text):

    if not text.strip():
        return "Please enter text", 0

    result = predict_sentiment(text)

    return (
        result["sentiment"],
        result["confidence"]
    )

demo = gr.Interface(
    fn=analyse,
    inputs=gr.Textbox(
        label="Customer Feedback"
    ),
    outputs=[
        gr.Textbox(label="Sentiment"),
        gr.Number(label="Confidence")
    ],
    title="Product Feedback Sentiment Analysis"
)

demo.launch(server_name="0.0.0.0")

* Running on local URL:  http://0.0.0.0:7860
* To create a public link, set `share=True` in `launch()`.


In [10]:
summary = (
    df["Sentiment"]
    .value_counts()
)

print(summary)

Sentiment
positive    93
negative    46
neutral     20
Name: count, dtype: int64


In [12]:
percentage = (
    df["Sentiment"]
    .value_counts(normalize=True) * 100
)

print(percentage)

Sentiment
positive    58.490566
negative    28.930818
neutral     12.578616
Name: proportion, dtype: float64
